# Failure Mode Analysis of Pedestrian Trajectory Prediction
## A Model-Agnostic Interpretable Risk Assessment Framework

**Author:** Pouya Bathaei Pourmand  
**Affiliation:** University of Genoa, Italy  
**Dataset:** ETH Pedestrian Dataset (public)

This notebook reproduces all experiments from the paper:
- Data loading and preprocessing
- Social Force Model (constant velocity baseline) prediction
- Failure definition and risk mapping
- Decision tree interpretability layer
- Cross-scene comparison (ETH vs Hotel)
- Figure generation

**Run in Google Colab** — all dependencies are pre-installed.

## 1. Setup: Download Dataset

In [ ]:
# Download OpenTraj repository which contains the ETH pedestrian dataset
# ETH dataset: real-world pedestrian trajectories recorded at 2.5 Hz
# Two scenes available: ETH (open campus) and Hotel (hotel entrance area)
!git clone https://github.com/crowdbotp/OpenTraj.git

import os
print('Available ETH dataset files:')
for root, dirs, files in os.walk('OpenTraj/datasets/ETH'):
    for file in files:
        print(os.path.join(root, file))

## 2. Load and Preprocess Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report

# ---------------------------------------------------------------
# Load ETH scene
# obsmat.txt columns: frame, ped_id, x, z, y, vx, vz, vy
# We use x, y (2D position) and vx, vy (2D velocity)
# ---------------------------------------------------------------
df_eth = pd.read_csv(
    'OpenTraj/datasets/ETH/seq_eth/obsmat.txt',
    sep=r'\s+', header=None,
    names=['frame', 'ped_id', 'x', 'z', 'y', 'vx', 'vz', 'vy']
)
df_eth = df_eth[['frame', 'ped_id', 'x', 'y', 'vx', 'vy']]
df_eth = df_eth.sort_values(['ped_id', 'frame']).reset_index(drop=True)

# Load Hotel scene
df_hotel = pd.read_csv(
    'OpenTraj/datasets/ETH/seq_hotel/obsmat.txt',
    sep=r'\s+', header=None,
    names=['frame', 'ped_id', 'x', 'z', 'y', 'vx', 'vz', 'vy']
)
df_hotel = df_hotel[['frame', 'ped_id', 'x', 'y', 'vx', 'vy']]
df_hotel = df_hotel.sort_values(['ped_id', 'frame']).reset_index(drop=True)

print(f'ETH scene   - Pedestrians: {df_eth.ped_id.nunique()}, Rows: {len(df_eth)}')
print(f'Hotel scene - Pedestrians: {df_hotel.ped_id.nunique()}, Rows: {len(df_hotel)}')

## 3. Social Force Model — Constant Velocity Baseline

We use a simplified Social Force Model (SFM) with constant velocity:
- Observe the first half of each trajectory
- Predict up to n_steps=8 future positions using the last observed velocity
- This isolates the effect of **initial conditions** on prediction error,
  independent of crowd interaction effects

The SFM equation used here:
$$p_{t+1} = p_t + v_t \cdot \Delta t, \quad v_t = 0.98 \cdot v_{t-1}$$

In [ ]:
def social_force_predict(group, n_steps=8):
    """
    Constant-velocity baseline predictor (simplified Social Force Model).

    Uses the first half of the trajectory as observation and predicts
    up to n_steps future positions using the last observed velocity.
    A slight damping factor (0.98) is applied per step.

    Parameters:
        group    (DataFrame): Single pedestrian trajectory (x, y, vx, vy)
        n_steps  (int)      : Maximum number of steps to predict (default=8)

    Returns:
        dict with observation, ground truth, prediction, and initial features
        None if trajectory is too short to evaluate
    """
    positions  = group[['x', 'y']].values
    velocities = group[['vx', 'vy']].values

    if len(positions) < 4:
        return None

    # Split trajectory: first half = observation, second half = ground truth
    obs_len  = len(positions) // 2
    obs_pos  = positions[:obs_len]
    obs_vel  = velocities[:obs_len]
    gt_pos   = positions[obs_len:obs_len + n_steps]  # cap at n_steps

    if len(gt_pos) < 2:
        return None

    # Initial state at prediction time (last observed position and velocity)
    curr_pos = obs_pos[-1].copy()
    curr_vel = obs_vel[-1].copy()
    speed    = np.linalg.norm(curr_vel)
    if speed < 0.01:
        speed = 0.5  # assign minimum speed to avoid zero-division

    # Predict future positions using constant velocity with slight damping
    dt        = 0.4   # ETH dataset timestep in seconds (2.5 Hz)
    predicted = []
    pos = curr_pos.copy()
    vel = curr_vel.copy()

    for _ in range(len(gt_pos)):
        vel = vel * 0.98  # slight velocity damping per step
        pos = pos + vel * dt
        predicted.append(pos.copy())

    return {
        'obs_pos'          : obs_pos,
        'gt_pos'           : gt_pos,
        'pred_pos'         : np.array(predicted),
        'init_speed'       : speed,
        'init_orientation' : np.arctan2(curr_vel[1], curr_vel[0]),
        'init_x'           : curr_pos[0],
        'init_y'           : curr_pos[1]
    }


def evaluate_scene(df, scene_name):
    """
    Run prediction and compute ADE/FDE for all pedestrians in a scene.
    Failure is defined as ADE > 75th percentile (data-driven threshold).

    Parameters:
        df          (DataFrame): Scene data with columns x, y, vx, vy
        scene_name  (str)      : Scene label for display

    Returns:
        results_df  (DataFrame): Per-trajectory results with failure label
        threshold   (float)    : ADE failure threshold (75th percentile)
    """
    results = []

    for ped_id, group in df.groupby('ped_id'):
        group  = group.sort_values('frame')
        result = social_force_predict(group)
        if result is None:
            continue

        gt      = result['gt_pos']
        pred    = result['pred_pos']
        min_len = min(len(gt), len(pred))
        if min_len < 2:
            continue

        # ADE: mean L2 distance over the prediction horizon
        ade = np.mean(np.sqrt(np.sum((gt[:min_len] - pred[:min_len])**2, axis=1)))
        # FDE: L2 distance at the final predicted position
        fde = np.sqrt(np.sum((gt[min_len-1] - pred[min_len-1])**2))

        results.append({
            'ped_id'           : ped_id,
            'ade'              : ade,
            'fde'              : fde,
            'init_speed'       : result['init_speed'],
            'init_orientation' : result['init_orientation'],
            'init_x'           : result['init_x'],
            'init_y'           : result['init_y'],
            'scene'            : scene_name
        })

    results_df = pd.DataFrame(results)

    # Failure threshold: 75th percentile of ADE — no manual tuning required
    threshold  = np.percentile(results_df['ade'], 75)
    results_df['failure'] = (results_df['ade'] > threshold).astype(int)

    print(f'\n{scene_name} Scene:')
    print(f'  Trajectories      : {len(results_df)}')
    print(f'  Mean ADE          : {results_df["ade"].mean():.3f} m')
    print(f'  Mean FDE          : {results_df["fde"].mean():.3f} m')
    print(f'  Failure threshold : {threshold:.3f} m (75th percentile)')
    print(f'  Success / Failure : {(results_df["failure"]==0).sum()} / {(results_df["failure"]==1).sum()}')

    return results_df, threshold


# Run evaluation on both scenes
results_eth,   threshold_eth   = evaluate_scene(df_eth,   'ETH')
results_hotel, threshold_hotel = evaluate_scene(df_hotel, 'Hotel')

## 4. Decision Tree — Interpretable Failure Rules

A decision tree classifier is trained on four input features:
- `init_speed`: initial walking speed (m/s)
- `init_orientation`: initial heading angle (radians)
- `init_x`, `init_y`: initial position (metres)

Depth is limited to 4 and minimum 10 samples per leaf to prevent overfitting.

In [ ]:
features = ['init_speed', 'init_orientation', 'init_x', 'init_y']

# Train decision tree on ETH scene
clf_eth = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
clf_eth.fit(results_eth[features], results_eth['failure'])

# Train decision tree on Hotel scene
clf_hotel = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
clf_hotel.fit(results_hotel[features], results_hotel['failure'])

print('ETH Decision Tree:')
print(f'  Accuracy: {clf_eth.score(results_eth[features], results_eth["failure"])*100:.1f}%')
print('  Feature Importances:')
for f, imp in zip(features, clf_eth.feature_importances_):
    print(f'    {f}: {imp:.3f}')

print('\nHotel Decision Tree:')
print(f'  Accuracy: {clf_hotel.score(results_hotel[features], results_hotel["failure"])*100:.1f}%')
print('  Feature Importances:')
for f, imp in zip(features, clf_hotel.feature_importances_):
    print(f'    {f}: {imp:.3f}')

print('\nETH Classification Report:')
print(classification_report(
    results_eth['failure'],
    clf_eth.predict(results_eth[features]),
    target_names=['Success', 'Failure']
))

## 5. Figure Generation
### Figure 1: Cross-Scene Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top left: ETH risk map (orientation vs speed)
ax = axes[0, 0]
ax.scatter(results_eth[results_eth['failure']==0]['init_orientation'],
           results_eth[results_eth['failure']==0]['init_speed'],
           c='green', alpha=0.5, s=25, label='Success')
ax.scatter(results_eth[results_eth['failure']==1]['init_orientation'],
           results_eth[results_eth['failure']==1]['init_speed'],
           c='red', alpha=0.7, s=35, label='Failure')
ax.set_xlabel('Initial Orientation (rad)')
ax.set_ylabel('Initial Speed (m/s)')
ax.set_title('ETH Scene — Risk Map')
ax.legend(fontsize=8)

# Top right: Hotel risk map (orientation vs speed)
ax = axes[0, 1]
ax.scatter(results_hotel[results_hotel['failure']==0]['init_orientation'],
           results_hotel[results_hotel['failure']==0]['init_speed'],
           c='green', alpha=0.5, s=25, label='Success')
ax.scatter(results_hotel[results_hotel['failure']==1]['init_orientation'],
           results_hotel[results_hotel['failure']==1]['init_speed'],
           c='red', alpha=0.7, s=35, label='Failure')
ax.set_xlabel('Initial Orientation (rad)')
ax.set_ylabel('Initial Speed (m/s)')
ax.set_title('Hotel Scene — Risk Map')
ax.legend(fontsize=8)

# Bottom left: ADE distribution comparison
ax = axes[1, 0]
ax.hist(results_eth['ade'],   bins=30, alpha=0.6, color='blue',   label='ETH')
ax.hist(results_hotel['ade'], bins=30, alpha=0.6, color='orange', label='Hotel')
ax.axvline(x=threshold_eth,   color='blue',   linestyle='--', linewidth=1.5,
           label=f'ETH threshold ({threshold_eth:.2f}m)')
ax.axvline(x=threshold_hotel, color='orange', linestyle='--', linewidth=1.5,
           label=f'Hotel threshold ({threshold_hotel:.2f}m)')
ax.set_xlabel('ADE (m)')
ax.set_ylabel('Count')
ax.set_title('ADE Distribution — ETH vs Hotel')
ax.legend(fontsize=8)

# Bottom right: feature importance comparison
ax = axes[1, 1]
x     = np.arange(len(features))
width = 0.35
ax.bar(x - width/2, clf_eth.feature_importances_,
       width, label='ETH', color='blue', alpha=0.7)
ax.bar(x + width/2, clf_hotel.feature_importances_,
       width, label='Hotel', color='orange', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(['Speed', 'Orientation', 'X pos', 'Y pos'])
ax.set_ylabel('Feature Importance')
ax.set_title('Feature Importance — ETH vs Hotel')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('comparison_scenes.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

### Figure 2: Decision Tree (ETH Scene)

In [ ]:
# Visualise the learned failure rules as a decision tree
# Blue nodes = Success, Orange nodes = Failure
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    clf_eth,
    feature_names=features,
    class_names=['Success', 'Failure'],
    filled=True, rounded=True, ax=ax, fontsize=9
)
plt.title('Decision Tree — Failure Rules\n(Social Force Model, ETH Dataset)', fontsize=13)
plt.savefig('decision_tree_sfm.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

### Figure 3: ADE Distribution and Feature Importance (ETH Scene)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: ADE distribution — success vs failure
ax = axes[0]
ax.hist(results_eth[results_eth['failure']==0]['ade'],
        bins=30, alpha=0.7, color='green', label='Success')
ax.hist(results_eth[results_eth['failure']==1]['ade'],
        bins=30, alpha=0.7, color='red', label='Failure')
ax.axvline(x=threshold_eth, color='black', linestyle='--',
           label=f'Threshold ({threshold_eth:.2f} m)')
ax.set_xlabel('ADE (m)')
ax.set_ylabel('Count')
ax.set_title('ADE Distribution\nSuccess vs Failure (ETH)')
ax.legend()

# Right: feature importances — highlight most important in dark blue
ax = axes[1]
importances = clf_eth.feature_importances_
colors = ['#2196F3' if i == np.argmax(importances) else '#90CAF9'
          for i in range(len(importances))]
bars = ax.barh(features, importances, color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Feature Importance\n(Decision Tree, ETH Scene)')
for bar, imp in zip(bars, importances):
    ax.text(bar.get_width() + 0.005,
            bar.get_y() + bar.get_height()/2,
            f'{imp:.3f}', va='center')

plt.tight_layout()
plt.savefig('analysis_sfm.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## 6. Download All Figures

In [ ]:
from google.colab import files
files.download('comparison_scenes.png')
files.download('decision_tree_sfm.png')
files.download('analysis_sfm.png')
print('All figures downloaded successfully.')